# M1 — Case type

Predicts `case_type` (incident vs service_request) from the **summary only**.
Word + character TF-IDF into a class-balanced logistic regression, evaluated on a **group-aware split** so identical summaries never straddle train/test.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
search = os.getcwd()
while search != os.path.dirname(search) and not os.path.exists(os.path.join(search, "case_pipeline.py")):
    search = os.path.dirname(search)
sys.path.insert(0, search)
import numpy as np, pandas as pd
import case_pipeline as P   # shared loading / splits / features / evaluation

In [ ]:
# Data and feature contract -------------------------------------------------
df      = P.load_clean()          # 1,713 audited, non-spam cases
FEATURES = ["case_summary"]       # summary text only (no tags — tags are leakage)
TARGET   = "case_type"
classes  = sorted(df[TARGET].unique())

# Group-aware split: duplicate summaries stay on one side of the split.
train, test = P.group_split(df)

# Fit the shared estimator (word + char TF-IDF -> logistic regression).
model = P.make_clf([P.t_word(), P.t_char()]).fit(train[FEATURES], train[TARGET])
pred  = model.predict(test[FEATURES])

# Normalised evaluation + majority baseline for context.
baseline = df[TARGET].value_counts(normalize=True).iloc[0]
metrics  = P.evaluate(test[TARGET], pred, classes, baseline=baseline)
metrics

In [ ]:
# Held-out predictions the app will display (no retraining needed at serve time).
proba = model.predict_proba(test[FEATURES])
heldout = test[["case_id", "case_summary", TARGET]].copy().reset_index(drop=True)
heldout["predicted"]  = pred
heldout["confidence"] = proba.max(axis=1).round(3)

# Refit on ALL data for deployment, then save the bundle.
deployed = P.make_clf([P.t_word(), P.t_char()]).fit(df[FEATURES], df[TARGET])
P.save_bundle("m1_case_type", {
    "model": deployed, "target": TARGET, "features": FEATURES, "classes": classes, "kind": "multiclass",
    "label": "Case type", "metrics": metrics, "test": heldout,
    "report": P.class_report(test[TARGET], pred, classes),
    "confusion": P.conf_matrix(test[TARGET], pred, classes),
    "meta": {"contributor": "A", "approach": "summary-only text classifier",
             "split": "group-on-summary", "inputs": "summary only"}})